# Document Analyzer using LLM

This is uses a LLM to analyze a document and identify key insights.

## Step 1: Extract Text from PDF

In [73]:
import pdfplumber

file_path = './files/google_terms_of_service_en_in.pdf'

output_text_file = './files/google_terms_of_service_en_in.txt'

with pdfplumber.open(file_path) as pdf:
    extracted_text = ""
    for page in pdf.pages:
        try:
            extracted_text += page.extract_text()
        except Exception as e:
            print(f"Error extracting text from page {page.page_number}: {e}")    

with open(output_text_file, "w") as text_file:
    text_file.write(extracted_text) 

print(f"Text saved to {output_text_file}")               

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Text saved to ./files/google_terms_of_service_en_in.txt


In [74]:
with open(output_text_file, "r") as text_file:
    text_content = text_file.read()
    print(text_content[:500])

GOOGLE TERMS OF SERVICE
Effective May 22, 2024 | Archived versions
What’s covered in these terms
We know it’s tempting to skip these Terms of
Service, but it’s important to establish what you
can expect from us as you use Google services,
and what we expect from you.
These Terms of Service re ect the way Google’s business works, the laws that apply to
our company, and certain things we’ve always believed to be true. As a result, these Terms
of Service help de ne Google’s relationship with you as


## Step 2: Summarize Document
Get high level overview of document using a pre-trained summarization model like t5-small

In [75]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

# Load summarization pipeline with a smaller model

token = os.getenv("HF_TOKEN")

device = "mps" if torch.backends.mps.is_available() else "cpu"

model_name = "google/gemma-2b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
model = AutoModelForCausalLM.from_pretrained(model_name, token=token).to(device)

Loading weights: 100%|██████████| 164/164 [00:00<00:00, 2322.69it/s, Materializing param=model.norm.weight]                              


In [99]:
# Summarize the extracted text
def summarize_text(text):
    input_text = f"summarize: {text.strip()}"
    
    inputs = tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True)

    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.inference_mode():
        summary_ids = model.generate(**inputs,          
                                    max_new_tokens=150, 
                                    length_penalty=2.0, 
                                    num_beams=1,
                                    early_stopping=True,)
        
        summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary


print("Summary: ", summarize_text(text_content))

Summary:  summarize: GOOGLE TERMS OF SERVICE
Effective May 22, 2024 | Archived versions
What’s covered in these terms
We know it’s tempting to skip these Terms of
Service, but it’s important to establish what you
can expect from us as you use Google services,
and what we expect from you.
These Terms of Service re ect the way Google’s business works, the laws that apply to
our company, and certain things we’ve always believed to be true. As a result, these Terms
of Service help de ne Google’s relationship with you as you interact with our services. For
example, these terms include the following topic headings:
What you can expect from us, which describes how we provide and develop our
services
What we expect from you, which establishes certain rules for using our services
Content in Google services, which describes the intellectual property rights to the
content you  nd in our services — whether that content belongs to you, Google, or
others
In case of problems or disagreements, which d

## Step 3: Split Document into Sentences and passage

In [100]:
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

# break all 
sentences = sent_tokenize(text_content)

passages = []

current_passage = ""

# combine sentences into passages
for sentence in sentences:
    if len(current_passage) + len(sentence) <= 1000:
        current_passage = (current_passage + " " + sentence).strip()

    else:
        passages.append(current_passage.strip())
        current_passage = sentence
    
print(passages)          

['GOOGLE TERMS OF SERVICE\nEffective May 22, 2024 | Archived versions\nWhat’s covered in these terms\nWe know it’s tempting to skip these Terms of\nService, but it’s important to establish what you\ncan expect from us as you use Google services,\nand what we expect from you. These Terms of Service re\x00ect the way Google’s business works, the laws that apply to\nour company, and certain things we’ve always believed to be true. As a result, these Terms\nof Service help de\x00ne Google’s relationship with you as you interact with our services.', 'For\nexample, these terms include the following topic headings:\nWhat you can expect from us, which describes how we provide and develop our\nservices\nWhat we expect from you, which establishes certain rules for using our services\nContent in Google services, which describes the intellectual property rights to the\ncontent you \x00nd in our services — whether that content belongs to you, Google, or\nothers\nIn case of problems or disagreements

[nltk_data] Downloading package punkt to /Users/inventmbp/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Step 4 - Generate questions from LLMs

In [102]:

def generate_questions_pipeline(passage, min_questions=3):
    input_text = f'''
    Generate only {min_questions} short questions
    Generate only questions and not answers. Do not include any explanations or additional text.
    Do not number the questions. 
    Do not add whitespaces.
    
    Passage: {passage}
    '''    

    # print("Input text for question generation: ", input_text)
    # Clean the input text by removing newlines and extra spaces and characters
    input_text = input_text.replace("\n", " ").strip()
    
    inputs = tokenizer(input_text, 
                              return_tensors="pt", 
                              max_length=1024, 
                              truncation=True, 
                              tie_word_embeddings=False
                            )
    
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(**inputs, 
                                    max_new_tokens=150, 
                                    num_beams=1,
                                    # no_repeat_ngram_size=3,
                                    # early_stopping=False,
                                    # repetition_penalty=2.5
                            )
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]    
    
    questions = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
    # print("Generated questions: ", questions)
    
    # questions = [q.strip() for q in questions.split(",") if q.strip()]
    
    return questions


In [86]:
from transformers import pipeline


qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-uncased-distilled-squad",
    device=0 if torch.backends.mps.is_available() else -1
)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 2376.79it/s, Materializing param=qa_outputs.weight]                                     


## Step 4 - Answer questions from passage

In [105]:
for index, passage in enumerate(passages[:10]):
    questions = generate_questions_pipeline(passage).split('\n')
    
    questions = [q.strip() for q in questions if q.strip()]
    
    print(f"Passage {index + 1}: \n {passage} \n")
    print("\n")
    for q in questions:
        answer = qa_pipeline(question=q, context=passage)['answer']
        print(f"Q: {q}")
        print(f"A: {answer.strip()} \n")
        
    print(f"\n {'-' * 50}\n")


Passage 1: 
 GOOGLE TERMS OF SERVICE
Effective May 22, 2024 | Archived versions
What’s covered in these terms
We know it’s tempting to skip these Terms of
Service, but it’s important to establish what you
can expect from us as you use Google services,
and what we expect from you. These Terms of Service re ect the way Google’s business works, the laws that apply to
our company, and certain things we’ve always believed to be true. As a result, these Terms
of Service help de ne Google’s relationship with you as you interact with our services. 



Q: What is the purpose of these Terms of Service?
A: help de ne Google’s relationship with you as you interact with our services 

Q: What are the laws that apply to Google?
A: the laws that apply to
our company 

Q: What are the things that Google has always believed to be true?
A: the laws that apply to
our company 


 --------------------------------------------------

Passage 2: 
 For
example, these terms include the following topic headings: